Import & Install Libraries

In [ ]:
!apt-get update -qq
!apt-get install -y osmium-tool
!pip install osmium

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import shapely
import tqdm
import osmium

from google.colab import drive
from shapely.geometry import Point
from geopandas.tools import clip

print("Packages loaded successfully")

Mount Colab

In [ ]:
drive.mount('/content/drive')

File Paths

In [ ]:
PBF_2015 = "/content/drive/MyDrive/MYPROJECTFILE/OSMClinicsMonthPbf/OSM-Zimbabwe-clinics-2015-09-01.pbf"
PBF_2023 = "/content/drive/MyDrive/MYPROJECTFILE/OSMClinicsMonthPbf/OSM-Zimbabwe-clinics-2023-09-01.pbf"

Transform Data Files

In [ ]:
!osmium tags-filter \
  "$PBF_2015" \
  n/amenity=clinic \
  n/amenity=hospital \
  -o "/content/drive/MyDrive/MYPROJECTFILE/OSMClinicsMonthPbf/clinics_2015.pbf" \
  --overwrite

In [ ]:
!osmium tags-filter \
  "$PBF_2023" \
  n/amenity=clinic \
  n/amenity=hospital \
  -o "/content/drive/MyDrive/MYPROJECTFILE/OSMClinicsMonthPbf/clinics_2023.pbf" \
  --overwrite

Export Clinics through Python

In [ ]:
class ClinicExtractor(osmium.SimpleHandler):

    def __init__(self):
        super().__init__()
        self.clinics = {}

    def node(self, n):

        amenity = n.tags.get("amenity")

        if amenity in ["clinic", "hospital"]:

            if n.location.valid():

                self.clinics[n.id] = (
                    n.lon,
                    n.lat
                )

Load the 2015 and 2023 Clinics

In [ ]:
c2015 = ClinicExtractor()
c2015.apply_file("/content/drive/MyDrive/MYPROJECTFILE/OSMClinicsMonthPbf/clinics_2015.pbf", locations=True)

c2023 = ClinicExtractor()
c2023.apply_file("/content/drive/MyDrive/MYPROJECTFILE/OSMClinicsMonthPbf/clinics_2023.pbf", locations=True)

Compute New Clinics

In [ ]:
clinics_2015_ids = set(c2015.clinics.keys())
clinics_2023_ids = set(c2023.clinics.keys())

new_ids = clinics_2023_ids - clinics_2015_ids

Convert to GeoDataFrame

In [ ]:
new_clinics_gdf = gpd.GeoDataFrame(
    {
        "id": list(new_ids),
        "geometry": [
            Point(c2023.clinics[i])
            for i in new_ids
        ]
    },
    crs="EPSG:4326"
)

Load 2015 Baseline Map

In [ ]:
base_clinics_gdf = gpd.GeoDataFrame(
    {
        "id": list(c2015.clinics.keys()),
        "geometry": [
            Point(c2015.clinics[i])
            for i in c2015.clinics.keys()
        ]
    },
    crs="EPSG:4326"
)

Load Electoral Districts

In [ ]:
districts = gpd.read_file("/content/drive/MyDrive/MYPROJECTFILE/ElectDistricts/Zim_Fixed_Constituencies.shp")

print(districts.head())
print(districts.crs)

Dissolve the Shapefile

In [ ]:
zimbabwe_boundary = districts.dissolve()

Reproject

In [ ]:
base_clinics_gdf = base_clinics_gdf.set_crs("EPSG:4326", allow_override=True)
new_clinics_gdf = new_clinics_gdf.set_crs("EPSG:4326", allow_override=True)
districts = districts.set_crs("EPSG:4326", allow_override=True)
zimbabwe_boundary = zimbabwe_boundary.set_crs("EPSG:4326", allow_override=True)

Clip Outside Layers

In [ ]:
country_polygon = zimbabwe_boundary.geometry.union_all()

In [ ]:
base_clinics_gdf = base_clinics_gdf[
    base_clinics_gdf.geometry.within(country_polygon)
].copy()

new_clinics_gdf = new_clinics_gdf[
    new_clinics_gdf.geometry.within(country_polygon)
].copy()

Create Map

In [ ]:
minx, miny, maxx, maxy = zimbabwe_boundary.total_bounds

x_margin = (maxx - minx) * 0.05
y_margin = (maxy - miny) * 0.001

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 14)
)

# Existing clinics in 2015
base_clinics_gdf.plot(
    ax=ax,
    color="#d9d9d9",
    markersize=8,
    alpha=0.9
)

# Clinics added by 2023
new_clinics_gdf.plot(
    ax=ax,
    color="#666666",
    markersize=12,
    alpha=1
)

# Constituency boundaries
districts.boundary.plot(
    ax=ax,
    color="#000000",
    linewidth=1
)

ax.set_xlim(minx - x_margin, maxx + x_margin)
ax.set_ylim(miny - y_margin, maxy + y_margin)

ax.set_axis_off()

plt.tight_layout()

plt.show()

Export Figure

In [ ]:
OUTPUT = "/content/drive/MyDrive/MYPROJECTFILE/Zimbabwe_Clinics_2015_2023.png"

fig.savefig(
    OUTPUT,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

print("Saved:", OUTPUT)